# Wallet Signal Strategy Exploration

This notebook explores a stronger alternative to naive copy-trading.

Core idea: use **wallets as market-state sensors**, not as direct trade templates. Instead of blindly mirroring each trade, rank wallets on training data, then look for **early opening buys** from strong wallets in test/live data, with filters on price regime and whether strong wallets have already taken the opposite side.


In [185]:
import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


In [186]:
END_DATE_TRAIN = datetime.date(2026, 2, 10)
TRADES_DIR = Path("../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../data/trade_signals_workspace")
WALLET_STATS_PATH = WORKSPACE_DIR / "wallet_stats.parquet"
SELECTED_WALLETS_PATH = WORKSPACE_DIR / "selected_wallets.parquet"
OPEN_BUYS_PATH = WORKSPACE_DIR / "open_buys_test.parquet"
EXEC_TAPE_PATH = WORKSPACE_DIR / "execution_tape_test.parquet"

trade_files = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(trade_files)} shards in {TRADES_DIR}")
print(f"Train cutoff: {END_DATE_TRAIN}")
print(f"Workspace dir: {WORKSPACE_DIR}")


Found 16 shards in ../data/polygon_trades_processed
Train cutoff: 2026-02-10
Workspace dir: ../data/trade_signals_workspace


## Dataset overview

The raw source lives in `data/polygon_trades_processed/*.parquet`. For faster iteration, the notebook reads precomputed workspace tables from `data/trade_signals_workspace/`.


In [187]:
dataset_summary = pd.Series({
    "workspace_exists": WORKSPACE_DIR.exists(),
    "wallet_stats_exists": WALLET_STATS_PATH.exists(),
    "selected_wallets_exists": SELECTED_WALLETS_PATH.exists(),
    "open_buys_exists": OPEN_BUYS_PATH.exists(),
    "exec_tape_exists": EXEC_TAPE_PATH.exists(),
    "raw_shards": len(trade_files),
})
dataset_summary


workspace_exists           True
wallet_stats_exists        True
selected_wallets_exists    True
open_buys_exists           True
exec_tape_exists           True
raw_shards                   16
dtype: object

## 1. Wallet skill persistence

A direct copy-trading strategy only works if wallet-level skill persists out of sample. We test that first.


In [188]:
wallet_stats = pd.read_parquet(WALLET_STATS_PATH)
paired_wallets = wallet_stats[(wallet_stats.train_trades >= 20) & (wallet_stats.test_trades >= 5)].copy()

pd.Series({
    "wallets_total": len(wallet_stats),
    "wallets_in_both": len(paired_wallets),
    "corr_train_pnl_vs_test_pnl": paired_wallets["train_pnl"].corr(paired_wallets["test_pnl"]),
    "corr_train_avg_pnl_vs_test_avg_pnl": paired_wallets["train_avg_pnl"].corr(paired_wallets["test_avg_pnl"]),
    "corr_train_hit_rate_vs_test_hit_rate": paired_wallets["train_hit_rate"].corr(paired_wallets["test_hit_rate"]),
})


wallets_total                           45407.000000
wallets_in_both                          7413.000000
corr_train_pnl_vs_test_pnl                  0.231373
corr_train_avg_pnl_vs_test_avg_pnl         -0.002092
corr_train_hit_rate_vs_test_hit_rate        0.386366
dtype: float64

In [189]:
paired_wallets["train_pnl_decile"] = pd.qcut(
    paired_wallets["train_pnl"].rank(method="first"),
    10,
    labels=False,
)

paired_wallets.groupby("train_pnl_decile").agg(
    wallets=("wallet", "size"),
    avg_train_pnl=("train_pnl", "mean"),
    avg_test_pnl=("test_pnl", "mean"),
    median_test_pnl=("test_pnl", "median"),
    avg_train_hit_rate=("train_hit_rate", "mean"),
    avg_test_hit_rate=("test_hit_rate", "mean"),
)


,wallets,avg_train_pnl,avg_test_pnl,median_test_pnl,avg_train_hit_rate,avg_test_hit_rate
train_pnl_decile,,,,,,
0,742,-8161.109459,-3672.070447,-188.225408,0.504288,0.493233
1,741,105.759285,-333.972888,-29.110072,0.516633,0.476763
2,741,324.636497,-457.577973,-18.352277,0.552785,0.489578
3,741,508.979340,-179.720930,-27.178010,0.563436,0.495520
4,742,781.830468,-355.256892,-54.195269,0.560950,0.487576
5,741,1241.489062,-102.557549,-10.500000,0.575784,0.510319
6,741,2055.685980,-123.060654,-71.312946,0.567297,0.482841
7,741,3838.630701,-1959.109012,-145.314148,0.582042,0.482660
8,741,9457.544935,-1619.073296,-77.574902,0.580182,0.503593


Observation: raw train PnL is a weak signal out of sample. Hit-rate persistence is much stronger than PnL persistence, so a better strategy should rank wallets by **stable directional accuracy**, not by headline profits alone.


## 2. Select wallets as signal generators

We use a shrunk hit-rate score to avoid over-ranking tiny-sample lucky wallets.


In [190]:
selected_wallets = pd.read_parquet(SELECTED_WALLETS_PATH)
selected_wallets[[
    "wallet", "train_trades", "train_pnl", "train_hit_rate", "hit_rate_shrunk", "train_volume", "score"
]].head(15)


,wallet,train_trades,train_pnl,train_hit_rate,hit_rate_shrunk,train_volume,score
0,0x751a2b86cab503496efd325c8344e10159349ea1,26057,157929.507671,0.973750,0.973568,4.408662e+07,51.494272
1,0xa16a1302ca05463f30faebeb5c045767fde233a1,46341,12912.785442,0.988606,0.988501,8.242064e+06,47.325101
2,0x336151559e8c8b048de5231dc8313e196b314363,24432,11870.316255,0.999673,0.999468,1.179087e+07,47.311502
3,0xd1c769317bd15de7768a70d0214cf0bbcc531d2b,7080,81935.043617,0.998023,0.997320,3.690157e+07,46.363396
4,0xc96aeabae8c81faf8d803201da1d2461cefc396a,17053,49052.448140,0.990148,0.989861,1.131857e+07,44.555129
5,0xf59db7ef18f784e17862c182d1134d5c8df38f85,14863,9837.394770,0.999933,0.999597,9.835643e+06,44.125732
6,0x06c0d07d5b59b25316bf1e6007aca6ad8f20bc23,20412,6185.647501,0.994513,0.994271,8.032349e+06,44.103578
7,0x7846e489e11c7d2b9d6c15a955c1df93d041c08d,17400,7920.649420,0.999713,0.999426,8.301740e+06,44.007496
8,0xa7ec2d5ce6c38557443a044e627d6abd317279fb,16014,1381.730220,0.999625,0.999314,7.534980e+06,43.155217
9,0x212af34bef48df3922c77ae109f67b690ede83cf,15842,773.449290,0.999684,0.999369,6.963815e+06,42.731264


## 3. Turn wallet trades into market-state events

For live trading, the interesting event is not just `BUY`, but **a strong wallet opening a fresh position** in a market outcome.

We derive:
- `open_buy`: wallet had zero prior position and started a new long
- `add_buy`: wallet already had a position and added more
- `reduce_sell`: wallet reduced the position


In [191]:
open_buys = pd.read_parquet(OPEN_BUYS_PATH)
open_buys["dt"] = pd.to_datetime(open_buys["dt"], utc=True)
open_buys.groupby("event_type").agg(
    rows=("wallet", "size"),
    avg_trade_pnl=("trade_pnl", "mean"),
    median_trade_pnl=("trade_pnl", "median"),
    avg_price=("price", "mean"),
)


,rows,avg_trade_pnl,median_trade_pnl,avg_price
event_type,,,,
open_buy,45434,2.295487,0.16025,0.91348


## 4. Consensus signal: same-side support vs opposite-side disagreement

A more interesting live signal is: when a strong wallet opens a position, how many other strong wallets have already opened the **same** outcome, and how many have already opened the **opposite** outcome in that market?

This captures whether the market is seeing early aligned conviction or expert disagreement.


In [192]:
open_buys["same_bucket"] = pd.cut(open_buys["prior_same"], bins=[-1, 0, 1, 2, 4, 9999], labels=["0", "1", "2", "3-4", "5+"])
open_buys["opp_bucket"] = pd.cut(open_buys["prior_opp"], bins=[-1, 0, 1, 2, 4, 9999], labels=["0", "1", "2", "3-4", "5+"])
open_buys[["market_key", "dt", "wallet", "price", "prior_same", "prior_opp", "same_bucket", "opp_bucket", "price_bucket"]].head(10)


,market_key,dt,wallet,price,prior_same,prior_opp,same_bucket,opp_bucket,price_bucket
0,0xb21d3671a9b3cd8048c4cfafc21e6a914d3e097c492f...,2026-02-11 00:01:31+00:00,0x8a37fe3a49a36de9facd04bd09ee379211f7c99c,0.998,0,0,0,0,"(0.9, 1.0]"
1,0x1ca1e6cdd4349b4fff1e712866eed4ef62d50e201664...,2026-02-11 00:06:15+00:00,0x205b4e3ab6e69e2250ccfdcf4e4613287966d66a,0.001,0,0,0,0,"(-0.001, 0.1]"
2,0xe901e6661232854f48ae15f23a8ec1ed4dadc828aa03...,2026-02-11 00:06:29+00:00,0xa49becb692927d455924583b5e3e5788246f4c40,0.820,0,0,0,0,"(0.75, 0.9]"
3,0x4581b6664fbf1d8ac6c4460d59cc9e30e7583421148c...,2026-02-11 00:08:25+00:00,0xdcd669c506a938bae9269a84b8bd2bdb220484cd,0.849,0,0,0,0,"(0.75, 0.9]"
4,0xf91604aae685d24b889529b101c599db739db5d883d3...,2026-02-11 00:08:47+00:00,0x24ded84d901a743cfee095f8082b0f9f647183cb,0.996,0,0,0,0,"(0.9, 1.0]"
5,0xd81e442b5dfb5edaf3ae7dbe88aa26a2c1bcd71c9f05...,2026-02-11 00:09:49+00:00,0xf7f0b0b1e9c0fe02ccad926916ee31aef74b912c,0.991,0,0,0,0,"(0.9, 1.0]"
6,0x4915c3719d2a810b030d594f32fd1c3212b0de6b6ab7...,2026-02-11 00:11:57+00:00,0xa16a1302ca05463f30faebeb5c045767fde233a1,0.999,0,0,0,0,"(0.9, 1.0]"
7,0xaf9d5c4e844ad0e9922e56123fbff4da473b9dd321ab...,2026-02-11 00:12:49+00:00,0x8454f39e7fa6957dfbf1ecc574820ada1c92331a,0.995,0,0,0,0,"(0.9, 1.0]"
8,0xcf2e532022a77d44b23c94338380a4a139ea25e63a4f...,2026-02-11 00:13:39+00:00,0xdf974b8db371482e34ff44b0fbfce11a0abf4653,0.490,0,0,0,0,"(0.4, 0.6]"
9,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c5694...,2026-02-11 00:14:37+00:00,0xee324b84546110f1fa5d4b2558edb179c3a4463a,0.730,0,0,0,0,"(0.6, 0.75]"


In [193]:
open_buys.groupby(["same_bucket", "price_bucket"], observed=False).agg(
    n=("roi_if_copy", "size"),
    mean_roi=("roi_if_copy", "mean"),
    median_roi=("roi_if_copy", "median"),
    win_rate=("token_winner", "mean"),
    avg_price=("price", "mean"),
).query("n >= 100").sort_values("mean_roi", ascending=False).head(25)


n  mean_roi  median_roi  win_rate  avg_price
same_bucket price_bucket                                                   
1           (-0.001, 0.1]    111  5.592843   -1.000000  0.081081   0.024477
0           (-0.001, 0.1]    949  1.742255   -1.000000  0.042150   0.012065
            (0.1, 0.25]      314  0.497290   -1.000000  0.299363   0.191299
            (0.6, 0.75]      847  0.074145    0.428571  0.723731   0.674338
1           (0.6, 0.75]      155  0.051944    0.408451  0.716129   0.684929
            (0.75, 0.9]      292  0.035343    0.176471  0.866438   0.837373
0           (0.75, 0.9]     1017  0.024233    0.162791  0.863324   0.843070
            (0.25, 0.4]      434  0.024180   -1.000000  0.347926   0.340166
            (0.9, 1.0]     25168  0.007720    0.010101  0.994278   0.986813
1           (0.9, 1.0]      7724  0.003662    0.001001  0.996893   0.993371
2           (0.9, 1.0]      2561  0.000604    0.001001  0.994143   0.993586
3-4         (0.9, 1.0]      1788 -0.003010    0.001001  0.991611   0.994465
5+          (0.9, 1.0]      1213 -0.007133    0.001001  0.983512   0.990106
0           (0.4, 0.6]      2017 -0.062277   -1.000000  0.483887   0.516750
1           (0.4, 0.6]       178 -0.076603   -1.000000  0.477528   0.510579

## 5. Candidate strategies

The two most interesting families are:

1. **Stable early opener**
   - price in `0.60-0.75`
   - first strong-wallet opener on that outcome (`prior_same == 0`)
   - no strong-wallet opposite opener yet (`prior_opp == 0`)
   - more stable, smaller edge

2. **Longshot discovery sleeve**
   - price in `0.10-0.25`
   - first strong-wallet opener
   - no opposite strong-wallet opener yet
   - very high expected ROI, but lottery-like distribution and needs diversification + small sizing


In [194]:
def evaluate_strategy(signals, mask, dedupe_by_market=True):
    s = signals[mask].copy().sort_values("dt")
    if dedupe_by_market:
        s = s.drop_duplicates("market_key", keep="first")

    return pd.Series({
        "signals": len(s),
        "mean_roi": s["roi_if_copy"].mean() if len(s) else np.nan,
        "median_roi": s["roi_if_copy"].median() if len(s) else np.nan,
        "win_rate": s["token_winner"].mean() if len(s) else np.nan,
        "avg_price": s["price"].mean() if len(s) else np.nan,
    })

strategy_masks = {
    "stable_mid_price_first": (open_buys["price"].between(0.60, 0.75, inclusive="left")) & (open_buys["prior_same"] == 0) & (open_buys["prior_opp"] == 0),
    "higher_conf_1to2_prior": (open_buys["price"].between(0.75, 0.90, inclusive="both")) & (open_buys["prior_same"].between(1, 2)) & (open_buys["prior_opp"] == 0),
    "lottery_early_longshot": (open_buys["price"].between(0.10, 0.25, inclusive="left")) & (open_buys["prior_same"] == 0) & (open_buys["prior_opp"] == 0),
    "broad_early_no_opp": (open_buys["price"].between(0.25, 0.90, inclusive="both")) & (open_buys["prior_same"] <= 1) & (open_buys["prior_opp"] == 0),
}

pd.DataFrame({name: evaluate_strategy(open_buys, mask) for name, mask in strategy_masks.items()}).T.sort_values("mean_roi", ascending=False)


,signals,mean_roi,median_roi,win_rate,avg_price
lottery_early_longshot,217.0,0.718748,-1.000000,0.327189,0.185111
stable_mid_price_first,812.0,0.061943,0.449275,0.703202,0.662248
higher_conf_1to2_prior,262.0,0.058668,0.176471,0.881679,0.832710
broad_early_no_opp,4041.0,0.007500,0.190476,0.616927,0.609410


## 6. Backtesting framework

The backtest uses **training data only for wallet selection**. Signal generation, execution, and realized PnL use **test data only**.

Execution assumptions:
- fixed stake per signal
- fill uses the next later executable test trade within a **10 minute horizon**
- executable price can come from the same token trade or, for binary markets, the opposite token via `1 - price`
- optional latency before the fill search starts
- slippage and fee applied on top of that executable price
- hold to resolution
- optional daily cap on new positions
- dedupe to the first qualifying signal per market/outcome


In [195]:
def normalize_execution_tape(execution_tape):
    tape = execution_tape.copy()

    if "exec_price_raw" not in tape.columns:
        if "price" in tape.columns:
            tape["exec_price_raw"] = tape["price"].astype(float)
        else:
            raise ValueError("Execution tape missing both exec_price_raw and price columns")

    if "exec_source" not in tape.columns:
        tape["exec_source"] = "same_token"

    if "available_qty" not in tape.columns:
        if "quantity" in tape.columns:
            tape["available_qty"] = tape["quantity"].astype(float)
        elif "available_usdc" in tape.columns:
            tape["available_qty"] = tape["available_usdc"].astype(float) / tape["exec_price_raw"].clip(lower=1e-9)
        else:
            raise ValueError("Execution tape missing available_qty. Rebuild with build_trade_signal_workspace.py")

    required = ["market_key", "tape_dt", "exec_price_raw", "exec_source", "available_qty"]
    missing = [c for c in required if c not in tape.columns]
    if missing:
        raise ValueError(f"Execution tape missing required columns: {missing}")

    return tape


def build_tape_groups(market_tape):
    return {k: g.reset_index(drop=True) for k, g in market_tape.groupby("market_key", sort=False)}


def attach_forward_fills(
    trades,
    tape_groups,
    latency_seconds=0,
    fill_horizon_seconds=600,
    slippage_bps=50.0,
    min_fill_ratio=1.0,
    ladder_price_bounds=None,
):
    trades = trades.copy().sort_values(["market_key", "dt"]).reset_index(drop=True)
    trades["fill_search_dt"] = trades["dt"] + pd.to_timedelta(latency_seconds, unit="s")
    trades["fill_deadline_dt"] = trades["fill_search_dt"] + pd.to_timedelta(fill_horizon_seconds, unit="s")

    slippage = slippage_bps / 10_000.0
    filled_parts = []

    for market_key, group in trades.groupby("market_key", sort=False):
        tape = tape_groups.get(market_key)
        if tape is None or tape.empty:
            continue

        tape = tape.sort_values("tape_dt").reset_index(drop=True).copy()
        if ladder_price_bounds is not None:
            low, high, inclusive = ladder_price_bounds
            tape = tape[tape["exec_price_raw"].between(low, high, inclusive=inclusive)].reset_index(drop=True)
            if tape.empty:
                continue
        tape_times = tape["tape_dt"].to_numpy()
        tape_prices = tape["exec_price_raw"].to_numpy(dtype=float)
        tape_qty_remaining = tape["available_qty"].to_numpy(dtype=float).copy()
        tape_source = tape["exec_source"].to_numpy()

        for row in group.itertuples(index=False):
            start_idx = tape_times.searchsorted(row.fill_search_dt, side="right")
            if start_idx >= len(tape):
                continue

            remaining_usdc = float(row.stake_usdc)
            filled_usdc = 0.0
            filled_qty = 0.0
            same_qty = 0.0
            opp_qty = 0.0
            fill_dt = None

            j = start_idx
            while j < len(tape) and tape_times[j] <= row.fill_deadline_dt and remaining_usdc > 1e-9:
                avail_qty = tape_qty_remaining[j]
                if avail_qty <= 1e-12:
                    j += 1
                    continue

                exec_price = float(np.clip(tape_prices[j] * (1.0 + slippage), 0.001, 0.999))
                max_usdc_here = avail_qty * exec_price
                take_usdc = min(remaining_usdc, max_usdc_here)
                take_qty = take_usdc / exec_price

                tape_qty_remaining[j] -= take_qty
                remaining_usdc -= take_usdc
                filled_usdc += take_usdc
                filled_qty += take_qty
                fill_dt = tape_times[j]

                if tape_source[j] == "same_token":
                    same_qty += take_qty
                else:
                    opp_qty += take_qty
                j += 1

            fill_ratio = filled_usdc / float(row.stake_usdc) if row.stake_usdc > 0 else 0.0
            if filled_usdc <= 0 or fill_ratio + 1e-12 < min_fill_ratio:
                continue

            out = pd.DataFrame([row._asdict()])
            out["filled_usdc"] = filled_usdc
            out["filled_qty"] = filled_qty
            out["exec_price"] = filled_usdc / filled_qty
            out["exec_price_raw"] = out["exec_price"]
            out["fill_ratio"] = fill_ratio
            out["tape_dt"] = fill_dt
            out["same_fill_share"] = same_qty / filled_qty if filled_qty > 0 else np.nan
            out["opposite_fill_share"] = opp_qty / filled_qty if filled_qty > 0 else np.nan
            out["exec_source"] = (
                "same_token" if opp_qty <= 1e-12 else ("opposite_token" if same_qty <= 1e-12 else "mixed")
            )
            filled_parts.append(out)

    if not filled_parts:
        return trades.iloc[0:0].copy()
    return pd.concat(filled_parts, ignore_index=True).sort_values("dt").reset_index(drop=True)


def backtest_strategy(
    signals,
    mask,
    tape_groups,
    stake_usdc=100.0,
    latency_seconds=0,
    fill_horizon_seconds=600,
    slippage_bps=50.0,
    fee_bps=10.0,
    max_signals_per_day=None,
    dedupe_by_market=True,
    starting_capital=10_000.0,
    min_fill_ratio=1.0,
    ladder_price_bounds=None,
):
    trades = signals[mask].copy().sort_values("dt")
    signal_count = len(trades)
    if dedupe_by_market:
        trades = trades.drop_duplicates("market_key", keep="first")
    trades["trade_date"] = trades["dt"].dt.floor("D")
    if max_signals_per_day is not None:
        trades["daily_rank"] = trades.groupby("trade_date").cumcount() + 1
        trades = trades[trades["daily_rank"] <= max_signals_per_day].copy()
    candidate_count = len(trades)
    trades["stake_usdc"] = float(stake_usdc)
    trades = attach_forward_fills(
        trades,
        tape_groups=tape_groups,
        latency_seconds=latency_seconds,
        fill_horizon_seconds=fill_horizon_seconds,
        slippage_bps=slippage_bps,
        min_fill_ratio=min_fill_ratio,
        ladder_price_bounds=ladder_price_bounds,
    )
    fee = fee_bps / 10_000.0
    trades["gross_roi"] = np.where(trades["token_winner"], 1.0 / trades["exec_price"] - 1.0, -1.0)
    trades["net_roi"] = trades["gross_roi"] - fee
    trades["gross_pnl_usdc"] = trades["filled_usdc"] * trades["gross_roi"]
    trades["net_pnl_usdc"] = trades["filled_usdc"] * trades["net_roi"]
    daily = (trades.groupby("trade_date").agg(trades=("market_key", "size"), stake_usdc=("filled_usdc", "sum"), gross_pnl_usdc=("gross_pnl_usdc", "sum"), net_pnl_usdc=("net_pnl_usdc", "sum")).reset_index().sort_values("trade_date"))
    if len(daily):
        daily["cum_net_pnl_usdc"] = daily["net_pnl_usdc"].cumsum()
        daily["equity_usdc"] = starting_capital + daily["cum_net_pnl_usdc"]
        daily["peak_equity_usdc"] = daily["equity_usdc"].cummax()
        daily["drawdown_usdc"] = daily["equity_usdc"] - daily["peak_equity_usdc"]
    else:
        daily = pd.DataFrame(columns=["trade_date", "trades", "stake_usdc", "gross_pnl_usdc", "net_pnl_usdc", "cum_net_pnl_usdc", "equity_usdc", "peak_equity_usdc", "drawdown_usdc"])
    summary = pd.Series({
        "signals": signal_count,
        "candidate_trades": candidate_count,
        "filled_trades": len(trades),
        "fill_rate": len(trades) / candidate_count if candidate_count else np.nan,
        "avg_fill_ratio": trades["fill_ratio"].mean() if len(trades) else np.nan,
        "markets": trades["market_key"].nunique(),
        "days_active": daily["trade_date"].nunique() if len(daily) else 0,
        "total_stake_usdc": trades["filled_usdc"].sum(),
        "gross_pnl_usdc": trades["gross_pnl_usdc"].sum(),
        "net_pnl_usdc": trades["net_pnl_usdc"].sum(),
        "net_roi_on_stake": trades["net_pnl_usdc"].sum() / trades["filled_usdc"].sum() if len(trades) else np.nan,
        "win_rate": trades["token_winner"].mean() if len(trades) else np.nan,
        "avg_signal_price": trades["price"].mean() if len(trades) else np.nan,
        "avg_exec_price": trades["exec_price"].mean() if len(trades) else np.nan,
        "avg_fill_delay_minutes": ((trades["tape_dt"] - trades["dt"]).dt.total_seconds().mean() / 60.0) if len(trades) else np.nan,
        "opposite_fill_share": trades["opposite_fill_share"].mean() if len(trades) else np.nan,
        "avg_trades_per_day": daily["trades"].mean() if len(daily) else 0.0,
        "max_drawdown_usdc": daily["drawdown_usdc"].min() if len(daily) else 0.0,
    })
    return summary, trades, daily


In [196]:
test_market_tape = pd.read_parquet(EXEC_TAPE_PATH)
test_market_tape["tape_dt"] = pd.to_datetime(test_market_tape["tape_dt"], utc=True)
test_market_tape = normalize_execution_tape(test_market_tape)
tape_groups = build_tape_groups(test_market_tape)

BACKTEST_CONFIG = {
    "stake_usdc": 100.0,
    "latency_seconds": 0,
    "fill_horizon_seconds": 600,
    "slippage_bps": 50.0,
    "fee_bps": 10.0,
    "max_signals_per_day": 20,
    "dedupe_by_market": True,
    "starting_capital": 10_000.0,
}

backtest_table = []
backtest_runs = {}
for strategy_name, strategy_mask in strategy_masks.items():
    summary, trades_bt, daily_bt = backtest_strategy(open_buys, strategy_mask, tape_groups=tape_groups, **BACKTEST_CONFIG)
    row = summary.to_dict()
    row["strategy"] = strategy_name
    backtest_table.append(row)
    backtest_runs[strategy_name] = {"summary": summary, "trades": trades_bt, "daily": daily_bt}

backtest_summary = pd.DataFrame(backtest_table).set_index("strategy")
backtest_summary.sort_values("net_roi_on_stake", ascending=False)


,signals,candidate_trades,filled_trades,fill_rate,avg_fill_ratio,markets,days_active,total_stake_usdc,gross_pnl_usdc,net_pnl_usdc,net_roi_on_stake,win_rate,avg_signal_price,avg_exec_price,avg_fill_delay_minutes,opposite_fill_share,avg_trades_per_day,max_drawdown_usdc
strategy,,,,,,,,,,,,,,,,,,
lottery_early_longshot,225.0,217.0,60.0,0.276498,1.0,60.0,24.0,6000.0,886.380142,880.380142,0.146730,0.316667,0.183817,0.293016,2.283889,0.183479,2.500000,-845.834147
higher_conf_1to2_prior,294.0,262.0,176.0,0.671756,1.0,176.0,28.0,17600.0,1249.388935,1231.788935,0.069988,0.897727,0.839511,0.861399,1.352652,0.242334,6.285714,-192.146083
stable_mid_price_first,816.0,467.0,252.0,0.539615,1.0,252.0,28.0,25200.0,1611.824595,1586.624595,0.062961,0.746032,0.669214,0.700188,1.881349,0.165386,9.000000,-607.225285
broad_early_no_opp,4469.0,568.0,180.0,0.316901,1.0,180.0,27.0,18000.0,620.001691,602.001691,0.033445,0.761111,0.667450,0.739465,2.023148,0.166619,6.666667,-330.132287


In [197]:
execution_sweep = []
for strategy_name, strategy_mask in strategy_masks.items():
    for latency_seconds in [0, 30, 120]:
        for slippage_bps in [0, 25, 50, 100]:
            summary, _, _ = backtest_strategy(
                open_buys,
                strategy_mask,
                tape_groups=tape_groups,
                stake_usdc=100.0,
                latency_seconds=latency_seconds,
                fill_horizon_seconds=600,
                slippage_bps=slippage_bps,
                fee_bps=10.0,
                max_signals_per_day=20,
                dedupe_by_market=True,
                starting_capital=10_000.0,
            )
            row = summary.to_dict()
            row["strategy"] = strategy_name
            row["latency_seconds"] = latency_seconds
            row["slippage_bps"] = slippage_bps
            execution_sweep.append(row)

pd.DataFrame(execution_sweep)[[
    "strategy", "latency_seconds", "slippage_bps", "filled_trades", "fill_rate",
    "net_pnl_usdc", "net_roi_on_stake", "win_rate", "avg_fill_delay_minutes",
    "opposite_fill_share", "max_drawdown_usdc",
]].sort_values(["net_roi_on_stake", "strategy", "latency_seconds", "slippage_bps"], ascending=[False, True, True, True])


,strategy,latency_seconds,slippage_bps,filled_trades,fill_rate,net_pnl_usdc,net_roi_on_stake,win_rate,avg_fill_delay_minutes,opposite_fill_share,max_drawdown_usdc
32,lottery_early_longshot,120,0,53.0,0.244240,1015.509963,0.191606,0.339623,4.683648,0.162835,-600.600000
33,lottery_early_longshot,120,25,53.0,0.244240,1001.201099,0.188906,0.339623,4.683019,0.162799,-600.600000
34,lottery_early_longshot,120,50,53.0,0.244240,986.963423,0.186220,0.339623,4.683019,0.162762,-600.600000
35,lottery_early_longshot,120,100,53.0,0.244240,958.789621,0.180904,0.339623,4.664780,0.162621,-600.600000
24,lottery_early_longshot,0,0,60.0,0.276498,910.304672,0.151717,0.316667,2.283889,0.183440,-843.572727
25,lottery_early_longshot,0,25,60.0,0.276498,895.305095,0.149218,0.316667,2.283889,0.183460,-844.706257
26,lottery_early_longshot,0,50,60.0,0.276498,880.380142,0.146730,0.316667,2.283889,0.183479,-845.834147
27,lottery_early_longshot,0,100,60.0,0.276498,850.810544,0.141802,0.316667,2.281667,0.183521,-848.073177
12,higher_conf_1to2_prior,0,0,176.0,0.671756,1320.599481,0.075034,0.897727,1.352652,0.242513,-191.136386
13,higher_conf_1to2_prior,0,25,176.0,0.671756,1275.828297,0.072490,0.897727,1.352652,0.242423,-191.642493


In [204]:
backtest_runs["stable_mid_price_first"]["daily"].head(10)


,trade_date,trades,stake_usdc,gross_pnl_usdc,net_pnl_usdc,cum_net_pnl_usdc,equity_usdc,peak_equity_usdc,drawdown_usdc
0,2026-02-11 00:00:00+00:00,10,1000.0,150.401111,149.401111,149.401111,10149.401111,10149.401111,0.000000
1,2026-02-12 00:00:00+00:00,11,1100.0,101.211243,100.111243,249.512354,10249.512354,10249.512354,0.000000
2,2026-02-13 00:00:00+00:00,8,800.0,214.542938,213.742938,463.255292,10463.255292,10463.255292,0.000000
3,2026-02-14 00:00:00+00:00,11,1100.0,120.358052,119.258052,582.513345,10582.513345,10582.513345,0.000000
4,2026-02-15 00:00:00+00:00,13,1300.0,28.849419,27.549419,610.062764,10610.062764,10610.062764,0.000000
5,2026-02-16 00:00:00+00:00,5,500.0,50.958961,50.458961,660.521725,10660.521725,10660.521725,0.000000
6,2026-02-17 00:00:00+00:00,11,1100.0,76.185468,75.085468,735.607193,10735.607193,10735.607193,0.000000
7,2026-02-18 00:00:00+00:00,8,800.0,36.358529,35.558529,771.165722,10771.165722,10771.165722,0.000000
8,2026-02-19 00:00:00+00:00,6,600.0,-186.909856,-187.509856,583.655865,10583.655865,10771.165722,-187.509856
9,2026-02-20 00:00:00+00:00,8,800.0,60.840090,60.040090,643.695956,10643.695956,10771.165722,-127.469766


In [199]:
backtest_runs["stable_mid_price_first"]["trades"][[
    "dt", "tape_dt", "condition_id", "outcome", "wallet", "price", "exec_price_raw", "exec_source", "exec_price",
    "prior_same", "prior_opp", "token_winner", "net_roi", "net_pnl_usdc",
]].head(20)


,dt,tape_dt,condition_id,outcome,wallet,price,exec_price_raw,exec_source,exec_price,prior_same,prior_opp,token_winner,net_roi,net_pnl_usdc
0,2026-02-11 00:14:37+00:00,2026-02-11 00:24:11+00:00,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c5694...,Yes,0xee324b84546110f1fa5d4b2558edb179c3a4463a,0.730,0.724743,mixed,0.724743,0,0,False,-1.001000,-100.100000
1,2026-02-11 00:15:37+00:00,2026-02-11 00:19:39+00:00,0x93903372f8cc532d0f564def91479c373fa9f139e88a...,Svitolina,0xf1528f12e645462c344799b62b1b421a6a4c64aa,0.700,0.703500,same_token,0.703500,0,0,False,-1.001000,-100.100000
2,2026-02-11 00:29:53+00:00,2026-02-11 00:29:57+00:00,0x4581b6664fbf1d8ac6c4460d59cc9e30e7583421148c...,OG,0xdcd669c506a938bae9269a84b8bd2bdb220484cd,0.673,0.669716,mixed,0.669716,0,0,True,0.492170,49.216981
3,2026-02-11 02:07:51+00:00,2026-02-11 02:08:07+00:00,0x54eb38f5bf423df4a8f8b3d722092465b7e02d6175fd...,Rockets,0xebc0df15e665da95bf819105b723ae90042ffa41,0.710,0.703500,same_token,0.703500,0,0,True,0.420464,42.046410
4,2026-02-11 02:37:01+00:00,2026-02-11 02:43:45+00:00,0x0490a755d1e2fb1e5c83ed4e2a7b72f45e17937c5c6f...,Clippers,0x991c22d8d9a44bf6686a1e30c2a3d471b64bf05b,0.640,0.562697,mixed,0.562697,0,0,True,0.776157,77.615691
5,2026-02-11 02:42:27+00:00,2026-02-11 02:43:19+00:00,0x6e2f87d5aad94b17ea35231d15286bbae3ee75256a7b...,No,0x134a63b764ac7b008356e8db1857db94e6b09e42,0.610,0.660130,mixed,0.660130,0,0,True,0.513853,51.385284
6,2026-02-11 03:23:13+00:00,2026-02-11 03:23:23+00:00,0x1e26201b0e133ea07aeb9425bf8945250155a23f17b5...,Clippers,0x0cbb0a19fe967a735f6659d2a39338161cea59fc,0.730,0.987540,same_token,0.987540,0,0,True,0.011617,1.161705
7,2026-02-11 05:17:59+00:00,2026-02-11 05:18:09+00:00,0x5a61d2fd0feef9fd403a045319785825b6ab085fa696...,Up,0x269ced0ede348b4b3f8fd8470b19f65508f79017,0.670,0.683400,same_token,0.683400,0,0,True,0.462272,46.227188
8,2026-02-11 06:14:49+00:00,2026-02-11 06:15:19+00:00,0x05d6ad9610722474d08fe69de4f70c80a86c5b3c39b9...,Invictus Gaming,0x52ecea7b3159f09db589e4f4ee64872fd0bba6f3,0.680,0.704208,same_token,0.704208,0,0,True,0.419035,41.903504
9,2026-02-11 08:17:23+00:00,2026-02-11 08:17:33+00:00,0x0f3651592f4858f6590afbc6808934cc744479a2c780...,Invictus Gaming,0x161eb16874e34f545991e774b4e1ac5b65f86ef0,0.690,0.713550,same_token,0.713550,0,0,True,0.400443,40.044349


## 7. Price bucket backtests

Evaluate the core early-opener/no-opposition rule across the full price ladder, including low-price and near-certain buckets.


In [200]:
bucket_specs = [
    ("0.00-0.10", 0.00, 0.10, "left"),
    ("0.10-0.25", 0.10, 0.25, "left"),
    ("0.25-0.40", 0.25, 0.40, "left"),
    ("0.40-0.60", 0.40, 0.60, "left"),
    ("0.60-0.75", 0.60, 0.75, "left"),
    ("0.75-0.90", 0.75, 0.90, "left"),
    ("0.90-1.00", 0.90, 1.00, "both"),
]

bucket_strategy_masks = {}
for label, low, high, inclusive in bucket_specs:
    bucket_strategy_masks[label] = (
        open_buys["price"].between(low, high, inclusive=inclusive)
        & (open_buys["prior_same"] == 0)
        & (open_buys["prior_opp"] == 0)
    )

bucket_rows = []
bucket_runs = {}
for bucket_name, bucket_mask in bucket_strategy_masks.items():
    low, high, inclusive = next(spec[1:] for spec in bucket_specs if spec[0] == bucket_name)
    summary, trades_bt, daily_bt = backtest_strategy(
        open_buys,
        bucket_mask,
        tape_groups=tape_groups,
        ladder_price_bounds=(low, high, inclusive),
        **BACKTEST_CONFIG,
    )
    row = summary.to_dict()
    row["bucket"] = bucket_name
    bucket_rows.append(row)
    bucket_runs[bucket_name] = {"summary": summary, "trades": trades_bt, "daily": daily_bt}

bucket_backtest_summary = pd.DataFrame(bucket_rows).set_index("bucket")
bucket_backtest_summary[[
    "signals", "filled_trades", "fill_rate", "net_pnl_usdc", "net_roi_on_stake",
    "win_rate", "avg_signal_price", "avg_exec_price", "avg_fill_delay_minutes",
    "opposite_fill_share", "max_drawdown_usdc",
]]


,signals,filled_trades,fill_rate,net_pnl_usdc,net_roi_on_stake,win_rate,avg_signal_price,avg_exec_price,avg_fill_delay_minutes,opposite_fill_share,max_drawdown_usdc
bucket,,,,,,,,,,,
0.00-0.10,692.0,3.0,0.006289,-300.300000,-1.001000,0.000000,0.071333,0.071119,2.666667,0.018373,-200.200000
0.10-0.25,225.0,35.0,0.161290,-345.661554,-0.098760,0.171429,0.182000,0.181521,2.126667,0.214006,-853.264975
0.25-0.40,343.0,126.0,0.376119,-2354.185424,-0.186840,0.261905,0.329810,0.330618,1.766138,0.146426,-2696.245878
0.40-0.60,1861.0,196.0,0.348135,-1038.688356,-0.052994,0.474490,0.496102,0.500994,1.529932,0.174754,-1507.942661
0.60-0.75,816.0,205.0,0.438972,1316.619745,0.064225,0.717073,0.667751,0.672018,1.759187,0.163537,-540.394751
0.75-0.90,828.0,182.0,0.340187,-77.510911,-0.004259,0.829670,0.823692,0.831132,2.005495,0.234140,-427.236502
0.90-1.00,24565.0,129.0,0.222414,-35.305587,-0.002737,0.992248,0.980054,0.993828,2.652972,0.061339,-99.408253


In [201]:
def write_bucket_pnl_svg(summary_df, output_path):
    plot_df = summary_df.reset_index().copy()
    plot_df["net_pnl_usdc"] = plot_df["net_pnl_usdc"].fillna(0.0)
    width = 900
    height = 420
    left = 80
    right = 30
    top = 40
    bottom = 90
    inner_w = width - left - right
    inner_h = height - top - bottom
    values = plot_df["net_pnl_usdc"].tolist()
    labels = plot_df["bucket"].tolist()
    vmin = min(values + [0.0])
    vmax = max(values + [0.0])
    if vmax == vmin:
        vmax = vmin + 1.0

    def y_scale(v):
        return top + inner_h * (1.0 - (v - vmin) / (vmax - vmin))

    zero_y = y_scale(0.0)
    bar_slot = inner_w / max(len(values), 1)
    bar_w = bar_slot * 0.65
    pieces = []
    pieces.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">')
    pieces.append('<rect width="100%" height="100%" fill="#ffffff"/>')
    pieces.append(f'<text x="{left}" y="24" font-size="18" font-family="Arial" fill="#111">Bucket Strategy Net PnL (10m horizon)</text>')
    pieces.append(f'<line x1="{left}" y1="{zero_y:.1f}" x2="{width-right}" y2="{zero_y:.1f}" stroke="#333" stroke-width="1.2"/>')
    for frac in [0.0, 0.25, 0.5, 0.75, 1.0]:
        v = vmin + frac * (vmax - vmin)
        y = y_scale(v)
        pieces.append(f'<line x1="{left}" y1="{y:.1f}" x2="{width-right}" y2="{y:.1f}" stroke="#e5e7eb" stroke-width="1"/>')
        pieces.append(f'<text x="{left-10}" y="{y+4:.1f}" text-anchor="end" font-size="11" font-family="Arial" fill="#555">{v:,.0f}</text>')

    for i, (label, value) in enumerate(zip(labels, values)):
        x = left + i * bar_slot + (bar_slot - bar_w) / 2
        y = y_scale(max(value, 0.0))
        h = abs(y_scale(value) - zero_y)
        color = "#1f77b4" if value >= 0 else "#d62728"
        pieces.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{max(h, 1):.1f}" fill="{color}" rx="3"/>')
        pieces.append(f'<text x="{x + bar_w/2:.1f}" y="{top + inner_h + 18}" text-anchor="middle" font-size="11" font-family="Arial" fill="#222">{label}</text>')
        pieces.append(f'<text x="{x + bar_w/2:.1f}" y="{(y - 6) if value >= 0 else (y_scale(value) + 14):.1f}" text-anchor="middle" font-size="10" font-family="Arial" fill="#222">{value:,.0f}</text>')

    pieces.append(f'<text x="{width/2:.1f}" y="{height-20}" text-anchor="middle" font-size="12" font-family="Arial" fill="#444">Price bucket</text>')
    pieces.append(f'<text x="20" y="{height/2:.1f}" transform="rotate(-90 20 {height/2:.1f})" text-anchor="middle" font-size="12" font-family="Arial" fill="#444">Net PnL (USDC)</text>')
    pieces.append("</svg>")
    output_path.write_text("".join(pieces))
    return output_path

bucket_plot_path = Path("wallet_signal_bucket_pnl.svg")
write_bucket_pnl_svg(bucket_backtest_summary, bucket_plot_path)
bucket_plot_path


PosixPath('wallet_signal_bucket_pnl.svg')

## 8. Plotly time series

Render cumulative PnL as interactive Plotly time series and open it in the browser renderer.


In [202]:
pio.renderers.default = "browser"

def with_zero_anchor(daily):
    if daily.empty:
        return daily

    first_date = daily["trade_date"].min()
    anchor = pd.DataFrame({
        "trade_date": [first_date - pd.Timedelta(days=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, daily], ignore_index=True).sort_values("trade_date").reset_index(drop=True)


def add_daily_traces(fig, daily_map, row, col):
    for name, daily in daily_map.items():
        if daily.empty:
            continue
        daily_plot = with_zero_anchor(daily)
        fig.add_trace(
            go.Scatter(
                x=daily_plot["trade_date"],
                y=daily_plot["cum_net_pnl_usdc"],
                mode="lines",
                name=name,
                hovertemplate="%{x|%Y-%m-%d}<br>%{fullData.name}<br>Cumulative PnL: %{y:.2f} USDC<extra></extra>",
            ),
            row=row,
            col=col,
        )


def build_strategy_sum_daily(daily_map):
    parts = []
    for name, daily in daily_map.items():
        if daily.empty:
            continue
        part = daily[["trade_date", "net_pnl_usdc"]].copy()
        part["strategy"] = name
        parts.append(part)

    if not parts:
        return pd.DataFrame(columns=["trade_date", "net_pnl_usdc", "cum_net_pnl_usdc"])

    combined = pd.concat(parts, ignore_index=True)
    summed = (
        combined.groupby("trade_date", as_index=False)
        .agg(net_pnl_usdc=("net_pnl_usdc", "sum"))
        .sort_values("trade_date")
        .reset_index(drop=True)
    )
    summed["cum_net_pnl_usdc"] = summed["net_pnl_usdc"].cumsum()
    return summed


strategy_daily_map = {name: run["daily"] for name, run in backtest_runs.items()}
bucket_daily_map = {name: run["daily"] for name, run in bucket_runs.items()}
strategy_sum_daily = build_strategy_sum_daily(strategy_daily_map)

pnl_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.10,
    subplot_titles=(
        "Strategy Cumulative PnL",
        "Price Bucket Cumulative PnL",
        "Sum of All Strategies (Cumulative PnL)",
    ),
)

add_daily_traces(pnl_fig, strategy_daily_map, row=1, col=1)
add_daily_traces(pnl_fig, bucket_daily_map, row=2, col=1)
if not strategy_sum_daily.empty:
    sum_plot = with_zero_anchor(strategy_sum_daily)
    pnl_fig.add_trace(
        go.Scatter(
            x=sum_plot["trade_date"],
            y=sum_plot["cum_net_pnl_usdc"],
            mode="lines",
            name="all_strategies_sum",
            line=dict(width=3),
            hovertemplate="%{x|%Y-%m-%d}<br>All Strategies Sum<br>Cumulative PnL: %{y:.2f} USDC<extra></extra>",
        ),
        row=3,
        col=1,
    )

pnl_fig.update_layout(
    title="Trade Signal PnL Time Series",
    template="plotly_white",
    height=1300,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    margin=dict(l=60, r=40, t=100, b=60),
)
pnl_fig.update_yaxes(title_text="Cumulative PnL (USDC)", row=1, col=1)
pnl_fig.update_yaxes(title_text="Cumulative PnL (USDC)", row=2, col=1)
pnl_fig.update_yaxes(title_text="Cumulative PnL (USDC)", row=3, col=1)
pnl_fig.update_xaxes(title_text="Date", row=1, col=1)
pnl_fig.update_xaxes(title_text="Date", row=2, col=1)
pnl_fig.update_xaxes(title_text="Date", row=3, col=1)

plotly_html_path = Path("trade_signal_pnl_timeseries.html")
pnl_fig.write_html(plotly_html_path, include_plotlyjs=True, auto_open=False)
pnl_fig.show(renderer="browser")
plotly_html_path


PosixPath('trade_signal_pnl_timeseries.html')

## 9. Interpretation

The forward-fill backtest is stricter than the earlier same-trade approximation, because entries are priced off the next later executable test trade within a 10 minute horizon.

What this means:
- wallet ranking is trained on `is_train == True` only
- every simulated fill and every realized PnL comes from `is_train == False` only
- same-token and opposite-token execution are both considered for binary markets
- fill rate now matters, because some signals do not find an executable trade quickly enough
- latency sensitivity can be measured directly with the execution sweep


## 10. Recommended live strategy

The strongest idea is **skill-weighted early consensus with opposition filter**.

Live rules:
- maintain a watchlist of top wallets ranked by shrunk train hit-rate, trade count, and volume
- for each market, track which watched wallets have opened each outcome
- trigger only on an `open_buy`, not every buy
- require `prior_opp == 0` so no watched wallet has already opened the other side
- use price regime as a sizing control:
  - `0.60-0.75` + first opener: main sleeve
  - `0.10-0.25` + first opener: optional small longshot sleeve
  - avoid crowded late entries where many watched wallets already piled in
- dedupe to one trade per market/outcome so multiple follow-up wallet entries do not cause overtrading

Why this is better than naive copy-trading:
- it uses persistent wallet **accuracy**, not fragile wallet PnL
- it trades on **information arrival** (fresh expert entry), not on wallet identity alone
- it filters out cases where experts are already split across outcomes
- it naturally supports live implementation from a trade stream
